# Les installations

In [ ]:
! pip install supabase

In [ ]:
! pip install chromadb

In [ ]:
!pip install google-genai

# Les Librairies

In [1]:
import os
import re
import dotenv
import chromadb
import trafilatura
import unicodedata
from google import genai
from google.genai import types
from supabase import create_client
from chromadb.utils import embedding_functions


# Initialisation

## Les variables d'environnement

In [2]:
dotenv.load_dotenv()

url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY")
Supabase_Client = create_client(url, key)

gemini_key = os.getenv("GEMINI_API_KEY")

## La liste de documents

In [3]:
# liste des url de la documentation de l'API YouTube Data v3
urls_documentation = [
    "https://developers.google.com/youtube/v3/docs?hl=fr",
    "https://developers.google.com/youtube/v3/docs/activities?hl=fr",
    "https://developers.google.com/youtube/v3/docs/activities/list?hl=fr",
    "https://developers.google.com/youtube/v3/docs/captions?hl=fr",
    "https://developers.google.com/youtube/v3/docs/captions/list?hl=fr",
    "https://developers.google.com/youtube/v3/docs/captions/insert?hl=fr",
    "https://developers.google.com/youtube/v3/docs/captions/update?hl=fr",
    "https://developers.google.com/youtube/v3/docs/captions/download?hl=fr",
    "https://developers.google.com/youtube/v3/docs/captions/delete?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channelBanners?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channelBanners/insert?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channels?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channels/list?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channels/update?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channelSections?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channelSections/list?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channelSections/insert?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channelSections/update?hl=fr",
    "https://developers.google.com/youtube/v3/docs/channelSections/delete?hl=fr",
    "https://developers.google.com/youtube/v3/docs/comments?hl=fr",
    "https://developers.google.com/youtube/v3/docs/comments/list?hl=fr",
    "https://developers.google.com/youtube/v3/docs/comments/insert",
    "https://developers.google.com/youtube/v3/docs/comments/update",
    "https://developers.google.com/youtube/v3/docs/comments/setModerationStatus",
    "https://developers.google.com/youtube/v3/docs/comments/delete",
    "https://developers.google.com/youtube/v3/docs/commentThreads",
    "https://developers.google.com/youtube/v3/docs/commentThreads/list",
    "https://developers.google.com/youtube/v3/docs/commentThreads/insert",
    "https://developers.google.com/youtube/v3/docs/i18nLanguages",
    "https://developers.google.com/youtube/v3/docs/i18nLanguages/list",
    "https://developers.google.com/youtube/v3/docs/i18nRegions",
    "https://developers.google.com/youtube/v3/docs/i18nRegions/list",
    "https://developers.google.com/youtube/v3/docs/members",
    "https://developers.google.com/youtube/v3/docs/members/list",
    "https://developers.google.com/youtube/v3/docs/membershipsLevels",
    "https://developers.google.com/youtube/v3/docs/membershipsLevels/list",
    "https://developers.google.com/youtube/v3/docs/playlistImages",
    "https://developers.google.com/youtube/v3/docs/playlistImages/list",
    "https://developers.google.com/youtube/v3/docs/playlistImages/insert",
    "https://developers.google.com/youtube/v3/docs/playlistImages/update",
    "https://developers.google.com/youtube/v3/docs/playlistImages/delete",
    "https://developers.google.com/youtube/v3/docs/playlistItems",
    "https://developers.google.com/youtube/v3/docs/playlistItems/list",
    "https://developers.google.com/youtube/v3/docs/playlistItems/insert",
    "https://developers.google.com/youtube/v3/docs/playlistItems/update",
    "https://developers.google.com/youtube/v3/docs/playlistItems/delete",
    "https://developers.google.com/youtube/v3/docs/playlists",
    "https://developers.google.com/youtube/v3/docs/playlists/list",
    "https://developers.google.com/youtube/v3/docs/playlists/insert",
    "https://developers.google.com/youtube/v3/docs/playlists/update",
    "https://developers.google.com/youtube/v3/docs/playlists/delete",
    "https://developers.google.com/youtube/v3/docs/search",
    "https://developers.google.com/youtube/v3/docs/search/list",
    "https://developers.google.com/youtube/v3/docs/subscriptions",
    "https://developers.google.com/youtube/v3/docs/subscriptions/list",
    "https://developers.google.com/youtube/v3/docs/subscriptions/insert",
    "https://developers.google.com/youtube/v3/docs/subscriptions/delete",
    "https://developers.google.com/youtube/v3/docs/thumbnails",
    "https://developers.google.com/youtube/v3/docs/thumbnails/set",
    "https://developers.google.com/youtube/v3/docs/videoAbuseReportReasons",
    "https://developers.google.com/youtube/v3/docs/videoAbuseReportReasons/list",
    "https://developers.google.com/youtube/v3/docs/videoCategories",
    "https://developers.google.com/youtube/v3/docs/videoCategories/list",
    "https://developers.google.com/youtube/v3/docs/videos",
    "https://developers.google.com/youtube/v3/docs/videos/list",
    "https://developers.google.com/youtube/v3/docs/videos/insert",
    "https://developers.google.com/youtube/v3/docs/videos/update",
    "https://developers.google.com/youtube/v3/docs/videos/rate",
    "https://developers.google.com/youtube/v3/docs/videos/getRating",
    "https://developers.google.com/youtube/v3/docs/videos/reportAbuse",
    "https://developers.google.com/youtube/v3/docs/videos/delete",
    "https://developers.google.com/youtube/v3/docs/watermarks",
    "https://developers.google.com/youtube/v3/docs/watermarks/set",
    "https://developers.google.com/youtube/v3/docs/watermarks/unset",
    "https://developers.google.com/youtube/v3/docs/errors",
    "https://developers.google.com/youtube/v3/docs/core_errors",
    "https://cloud.google.com/apis/docs/system-parameters"
]

# Les fonctions 

## Scraping

In [4]:
def scraper_page(url: str, id ) -> dict | None:
    """_summary_
    Télécharge une page et extrait son contenu textuel.
    Retourne None si l'extraction échoue.

    Args:
        url (str): URL de la page à scraper
        id (int): Identifiant unique pour la page

    Returns:
        dict | None: Dictionnaire contenant l'ID, l'URL, le titre et le texte extrait, ou None si l'extraction échoue.
    """
    # Télécharger le HTML
    html = trafilatura.fetch_url(url)
    if not html:
        print(f"Échec téléchargement: {url}")
        return None

    # Extraire le contenu (sans commentaires, avec tableaux)
    texte = trafilatura.extract(
        html,
        include_comments=True,
        include_tables=True,
        output_format="txt"
    )

    if not texte:
        print(f"Pas de contenu extractible: {url}")
        return None

    # Extraire un titre depuis l'URL
    titre = url[41:].replace("?hl=fr", "").replace("/", "_")

    return {
        "id": id,
        "url": url,
        "titre": titre,
        "texte": texte
    }

## Nettoyage

In [5]:
def nettoyage(texte: str) -> str:
    """
    Nettoie le texte en normalisant les caractères Unicode, en supprimant les espaces multiples, les lignes vides excessives et les barres obliques inverses.

    Args:
        texte (str): Le texte à nettoyer.

    Returns:
        str: Le texte nettoyé.
    """
    texte = unicodedata.normalize("NFC", texte)
    texte = re.sub(r'[ \t]+', ' ', texte)  # Espaces multiples
    texte = re.sub(r'\n{3,}', '\n\n', texte)  # Lignes vides excessives
    texte = re.sub(r'\\', '', texte)  # supprime les \
    texte = texte.strip()
    return texte

## Chunking

In [15]:
# methode de chunking fixe
def chunker_fixe(chunk_size:int, overlap:int, texte:str) -> list:
    """_summary_
    Découpe un texte en segments/ chunks de taille fixe avec chevauchement.

    Args:
        chunk_size (int): Taille maximale de chaque chunk (en nombre de mots).
        overlap (int): Nombre de mots qui se chevauchent entre les chunks.
        texte (str): Le texte à découper.

    Returns:
        list: Liste de chunks (segments de texte).
    """
    assert chunk_size > overlap, "La taille du chunk doit être supérieure au chevauchement."

    mots = texte.split()

    chunks = []
    start = 0
    step = chunk_size - overlap

    for i in range(start, len(mots), step):
        chunk = " ".join(mots[i:i + chunk_size])
        chunks.append(chunk)
        #print(chunk)

        if start + step >= len(mots):
            break
        start += step
    
    return chunks

# implementation du chunking
def make_chuns(documents_bruts: list)-> tuple[list, list, list]:
    """_summary_

    Args:
        documents_bruts (list): Liste de dictionnaires contenant les documents bruts avec leurs métadonnées, issus du scraping.

    Returns:
        tuple[list, list, list]: Trois listes : 
            - liste_ids : Liste des IDs uniques pour chaque chunk.
            - liste_documents : Liste des textes des chunks.
            - liste_metadatas : Liste des métadonnées associées à chaque chunk.
    """

    liste_ids = []
    liste_documents = []
    liste_metadatas = []

    print("Application du chunking et préparation de l'indexation...")
    chunk_size = 200
    overlap = 50
    for doc in documents_bruts:
        # On applique ta fonction de découpage sur le texte brut du document
        chunks_du_document = chunker_fixe(chunk_size,overlap,nettoyage(doc["texte"]))
        
        for index, texte_du_chunk in enumerate(chunks_du_document):
            # print(f"Document ID: {doc['id']}, Chunk {index}: {texte_du_chunk[:60]}...")
            # Création d'un ID unique par chunk (ex: doc_1_chunk_000, doc_1_chunk_001...)
            id_unique_chunk = f"doc_{doc['id']}_chunk_{index:03d}"
            
            liste_ids.append(id_unique_chunk)
            liste_documents.append(texte_du_chunk) # Le texte que Chroma va vectoriser
            
            # On sauvegarde les métadonnées pour que le LLM sache d'où vient l'info
            liste_metadatas.append({
                "document_parent_id": doc["id"],
                "url": doc["url"],
                "titre": doc["titre"],
                "chunk_index": index
            })
    return liste_ids, liste_documents, liste_metadatas

## Embedding et stockage

In [8]:
# configuration de chroma avec un modèle d'embedding multilingue et création d'une collection
def chroma_config():
    """_summary_

    """

    # client local non persistant
    chroma_client = chromadb.Client()

    # config embedding mutilingue
    fonction_multilingue = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="paraphrase-multilingual-MiniLM-L12-v2")
    
    # creat a collection for our documents, embeddings and metadatas 
    collection = chroma_client.get_or_create_collection(name="youtube_api_docs",
                                        embedding_function=fonction_multilingue)
    
    return collection

# injection des chunks dans la collection chroma
def indexation_chroma(liste_ids, liste_documents, liste_metadatas, collection):
    """_summary_
    Injecte les chunks vectorisés dans la collection Chroma.

    Args:
        liste_ids (list): liste des IDs uniques pour chaque chunk.
        liste_documents (list): liste des textes des chunks.
        liste_metadatas (list): liste des métadonnées associées à chaque chunk.
    """

    if liste_ids:
        print(f"Vectorisation multilingue et injection de {len(liste_ids)} chunks dans Chroma...")
        try:
            # add dociuments to the collection  
            collection.add(
                ids= liste_ids,
                documents= liste_documents,
                metadatas= liste_metadatas
            )
            print("Les documents Supabase sont vectorisés dans Chroma.")
        except Exception as e:
            print(f"Erreur lors de l'injection Chroma : {e}")
    else:
        print("Aucun chunk généré")
    

## Requête - recherche - génération

In [25]:
# recherche de documents pour contexte dans la collection chroma
def get_contexte (query:list, n:int=3) -> str:
    """_summary_
    la fonction get_contexte interroge la collection Chroma pour récupérer les documents les plus pertinents en fonction de la requête de l'utilisateur. 
    Elle retourne le contexte sous forme de texte concaténé et les métadonnées associées.

    Args:
        query (list): Liste de chaînes de caractères représentant la requête de l'utilisateur.
        n (int): Nombre de résultats pertinents à récupérer (par défaut 3).
    """

    # vérifie que querry est une liste de chaînes de caractères
    assert isinstance(query, list), "Query doit être une liste."
    assert all(isinstance(item, str) for item in query), "Tous les éléments de la liste query doivent être des chaînes de caractères."
    # vérifie que query n'est pas vide
    assert len(query) > 0, "La liste de requête ne peut pas être vide."
    # vérifie que n est un entier positif superieur à 1
    assert isinstance(n, int) and n > 1, "n doit être un entier positif supérieur à 0."

    # on récupère la collection Chroma
    client = chromadb.Client()
    collection = client.get_collection(name="youtube_api_docs")

    # recherche des docuement pertinents pour la requête
    resultats = collection.query(
        query_texts=query,
        n_results=n
    )

    chunks_ = resultats["documents"][0]
    metadatas_= resultats["metadatas"][0]

    contexte = "\n\n".join(
        [f"[Extrait {i+1} (Source: {meta['url']})]\n{texte}" 
        for i, (texte, meta) in enumerate(zip(chunks_, metadatas_))]
    )

    return contexte, metadatas_

# augmentation du prompt avec le contexte et la question du développeur
def make_augmented_promt(contexte:str, query:str) -> str:
    """_summary_
    renvoie un prompt augmenté en combinant le contexte de la documentation et la question du développeur.

    Args:
        contexte (str): l'ensemble des extraits de documentation pertinents pour la question du développeur.
        query (str): la question posée par le développeur.
        Returns:
    """


    # Le prompt final fusionne le contexte et la question
    prompt_augmented = f"""CONTEXTE DE LA DOCUMENTATION OFFICIELLE :
    {contexte}

    QUESTION DU DÉVELOPPEUR :
    {query}
    """

    return prompt_augmented

# interrogation du modèle Gemini pour générer une réponse à partir du prompt augmenté
def rag_executer(prompt:str, temperature:float=0.5) -> str:
    """_summary_
    Interroge le modèle Gemini pour générer une réponse à partir du prompt augmenté.
    
    Args:
        prompt (str): Le prompt augmenté contenant le contexte et la question du développeur.
    
    Returns:
        reponse (str) : La réponse générée par le modèle Gemini.
    
    """

    system_instruction = """Tu es un ingénieur support technique expert. Ton but est d'aider les développeurs en répondant à leurs questions.
    Consignes strictes :
    1. Réponds en utilisant UNIQUEMENT les informations du contexte fourni.
    2. Si la réponse n'est pas dans le contexte, dis clairement et poliment : "Je suis désolé, mais la documentation fournie ne contient pas l'information pour répondre à cette question."
    3. Ne pas inventer, extrapoler ou utiliser tes connaissances générales. Réponds TOUJOURS en français.
    """

    client_gemini = genai.Client()

    reponse = client_gemini.models.generate_content(
        model = "gemini-3.5-flash",
        contents =prompt,
        config=types.GenerateContentConfig(system_instruction=system_instruction, temperature=temperature)

    )

    return reponse

# implémentation de la fonction principale pour exécuter le RAG
def main_rag(query:str, n:int=3, temperature:float=0.5):
    """_summary_
    Implémente le processus de RAG (Retrieval-Augmented Generation) pour répondre à une question posée par un développeur.

    Args:
        query (str): La question posée par le développeur.
        n (int): Le nombre de documents pertinents à récupérer pour le contexte (par défaut 3).
    """
    contexte, metadatas = get_contexte(query=[query], n=n)

    prompt_augmented = make_augmented_promt(contexte, query)

    response = rag_executer(prompt_augmented, temperature=temperature)

    print("\n" + "="*40)
    print(f"Votre question:\n{query}")
    print("\n" + "="*40)
    print("RÉPONSE DU RAG :")
    print("="*40)
    print(response.text)
    print("="*40)

    print("\n🔗 SOURCES UTILISÉES :")
    urls_uniques = set([meta['url'] for meta in metadatas])
    for url in urls_uniques:
        print(f"- {url}")


# Chargements de document dans une base de données

Le code ci-dessous ne s'éxécute qu'une seule fois pour scraper les pages de l'API puis stocker les docuements dans la base de données Supabase. 
Pour utiliser cette base de données, il faut d'abord créer un compte pour obtenir le clé api et l'url supabase permettant de se connecter à la base de données.

In [ ]:
# for i,doc in enumerate(urls_documentation):
#     # definition d'un id dynamique pour chaque document
#     doc_id = i + 1
#     # print(f"Traitemen du document {doc_id}: {doc}")

#     # fonction de scraping
#     doc_info = scraper_page(doc, doc_id)
#     # print(f"Longueur du texte extrait: {doc_info['longueur']} caractères")

#     # si le scraping a réussi, insertion dans Supabase
#     if doc_info is not None:
#         try :
#             # response = supabase.api_client.table("api_docs").insert(doc_info).execute()
#             response = (Supabase_Client.table("api_docs")
#                 .insert(doc_info)
#                 .execute()
#             )
#             # print(f"Document inséré avec ID: {response.data[0]['id']}")
#         except Exception as e:
#             print(f"Erreur d'insertion Supabase pour {doc}: {e}")
#     else:
#         print(f"Document {doc_id} vide car le scraping a échoué.")

# Exécution du RAG 

In [16]:
# on verfiie que les données ont bien été insérées
try :
    response = Supabase_Client.table("api_docs").select("id", "texte", "url", "titre").execute()
    print(f"Nombre de documents insérés: {len(response.data)}")
    print(f"Nombre de documents scrapé : {len(urls_documentation)}")
except Exception as e:
    print(f"Erreur de récupération des données : {e}")

Nombre de documents insérés: 77
Nombre de documents scrapé : 77


In [17]:
try :
    collection = chroma_config()
    try :
        liste_ids, liste_documents, liste_metadatas = make_chuns(documents_bruts=response.data)
        try :
            indexation_chroma(liste_ids, liste_documents, liste_metadatas,collection)
        except Exception as e:
            print(f"Erreur lors de l'indexation Chroma : {e}")
    except Exception as e:
        print(f"Erreur lors de make_chunks: {e}")
except Exception as e:
    print(f"Erreur lors de la configuration Chroma : {e}")

Application du chunking et préparation de l'indexation...
🚀 Vectorisation multilingue et injection de 522 chunks dans Chroma...
🎉 Terminé ! Tes documents Supabase sont maintenant vectorisés dans Chroma.


# Interrogation du RAG

In [18]:
query = ["Comment paramétrer les subtitles ?"]

main_rag(query[0])

Votre question:
Comment paramétrer les subtitles ?

✨ RÉPONSE DU RAG :
D'après la documentation fournie, voici comment vous pouvez paramétrer les sous-titres (captions) :

### 1. Lors du téléchargement d'une piste de sous-titres
Vous pouvez utiliser les paramètres de requête suivants :
* **`tfmt`** : Spécifie que la piste de sous-titres doit être renvoyée dans un format spécifique. Si ce paramètre n'est pas inclus, le sous-titre est renvoyé dans son format d'origine.
* **`tlang`** : Spécifie que la réponse doit renvoyer une traduction automatique de la piste (générée par exemple avec Google Traduction). La valeur doit être un code de langue ISO 639-1 à deux lettres.

### 2. Configuration des propriétés de la piste de sous-titres (dans le `snippet`)
Vous pouvez définir les propriétés suivantes pour configurer une piste de sous-titres :
* **Nom** : La longueur maximale du nom du sous-titre est de 150 caractères.
* **`snippet.audioTrackType`** (string) : Définit le type de piste audio ass

In [27]:
query = ["Comment paramétrer les subtitles ?"]

main_rag(query[0], n=2, temperature=0.7)


Votre question:
Comment paramétrer les subtitles ?

RÉPONSE DU RAG :
D'après la documentation fournie, pour paramétrer la récupération d'une piste de sous-titres, vous pouvez utiliser les paramètres suivants lors de l'appel de la méthode :

*   **`tfmt`** : Ce paramètre spécifie que la piste de sous-titres doit être renvoyée dans un format spécifique. Si ce paramètre n'est pas inclus dans la requête, le sous-titre est renvoyé dans son format d'origine.
*   **`tlang`** (chaîne de caractères) : Ce paramètre spécifie que la réponse de l'API doit renvoyer une traduction de la piste de sous-titres spécifiée. La valeur à fournir est un code de langue ISO 639-1 à deux lettres (qui identifie la langue souhaitée). La traduction est générée automatiquement (par exemple via Google Traduction).

**Consigne supplémentaire pour la requête :**
*   Vous ne devez pas fournir de corps de requête (Request body) lors de l'appel de cette méthode.

🔗 SOURCES UTILISÉES :
- https://developers.google.com/yout

In [20]:
query = ["Comment paramétrer les subtitles ?"]

main_rag(query[0], n=2, temperature=1.3)

Votre question:
Comment paramétrer les subtitles ?

✨ RÉPONSE DU RAG :
D'après la documentation fournie (Extrait 1), pour paramétrer la récupération (le téléchargement) des sous-titres, vous pouvez utiliser les paramètres suivants :

*   **`tfmt`** : Ce paramètre spécifie que la piste de sous-titres doit être renvoyée dans un format spécifique. Si vous n'incluez pas ce paramètre dans la requête, les sous-titres sont renvoyés dans leur format d'origine.
*   **`tlang`** (chaîne de caractères / *string*) : Ce paramètre spécifie que la réponse de l'API doit renvoyer une traduction automatique (générée par exemple par Google Traduction) de la piste de sous-titres. La valeur à fournir doit être un code de langue ISO 639-1 à deux lettres correspondant à la langue souhaitée.

**Consignes supplémentaires pour la requête :**
*   **Corps de la requête :** Vous ne devez pas fournir de corps de requête lors de l'appel de cette méthode.
*   **Réponse :** En cas de succès, la méthode renvoie un fichi

In [21]:
query = ["A quoi sert l'API YouTube ?"]

main_rag(query[0], n=2, temperature=0.7)

Votre question:
A quoi sert l'API YouTube ?

✨ RÉPONSE DU RAG :
Selon la documentation fournie, l'API YouTube Data vous permet d'intégrer à votre propre site Web ou application des fonctions qui sont normalement exécutées sur le site Web YouTube. 

Elle permet notamment de récupérer différents types de ressources (telles qu'une vidéo, une playlist ou un abonnement) et prend en charge des méthodes pour insérer, mettre à jour ou supprimer plusieurs de ces ressources.

🔗 SOURCES UTILISÉES :
- https://developers.google.com/youtube/v3/docs?hl=fr
- https://developers.google.com/youtube/v3/docs/channels?hl=fr


In [22]:
query = ["comment récupérer les commentaires"]

main_rag(query[0], n=5, temperature=0.7)

Votre question:
comment récupérer les commentaires

✨ RÉPONSE DU RAG :
D'après la documentation fournie, voici comment récupérer les commentaires :

1. **Pour récupérer une liste de commentaires :** Vous devez utiliser la méthode **`liste`** (ou `list`) sur les ressources `comments`. Cette méthode renvoie une liste de commentaires correspondant aux paramètres de la requête API.
2. **Pour récupérer toutes les réponses à un commentaire de premier niveau :** Vous devez appeler la méthode **`comments.list`** en utilisant le paramètre de requête **`parentId`** afin d'identifier le commentaire pour lequel vous souhaitez obtenir les réponses.

🔗 SOURCES UTILISÉES :
- https://developers.google.com/youtube/v3/docs/commentThreads/insert
- https://developers.google.com/youtube/v3/docs/comments/insert
- https://developers.google.com/youtube/v3/docs/comments?hl=fr
- https://developers.google.com/youtube/v3/docs/commentThreads


In [23]:
query = ["donne moi le code python pour récupérer les commentaires d'une vidéo"]

main_rag(query[0], n=5, temperature=0.7)

Votre question:
donne moi le code python pour récupérer les commentaires d'une vidéo

✨ RÉPONSE DU RAG :
Je suis désolé, mais la documentation fournie ne contient pas l'information pour répondre à cette question.

🔗 SOURCES UTILISÉES :
- https://developers.google.com/youtube/v3/docs/search/list
- https://developers.google.com/youtube/v3/docs?hl=fr
- https://developers.google.com/youtube/v3/docs/comments?hl=fr
- https://developers.google.com/youtube/v3/docs/videos/reportAbuse
- https://developers.google.com/youtube/v3/docs/playlistItems


In [24]:
query = ["donne moi la recette de la tarte aux pommes"]

main_rag(query[0], n=5, temperature=2)

Votre question:
donne moi la recette de la tarte aux pommes

✨ RÉPONSE DU RAG :
Je suis désolé, mais la documentation fournie ne contient pas l'information pour répondre à cette question.

🔗 SOURCES UTILISÉES :
- https://developers.google.com/youtube/v3/docs/playlists
- https://developers.google.com/youtube/v3/docs?hl=fr
- https://developers.google.com/youtube/v3/docs/channelSections?hl=fr
- https://developers.google.com/youtube/v3/docs/videos
- https://developers.google.com/youtube/v3/docs/playlistItems
